### Import libraries

In [0]:
import hashlib
import json
import logging
import re
import time
import uuid

from datetime import datetime, timedelta, timezone
from urllib.parse import urljoin, urlparse

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from delta.tables import DeltaTable
from pyspark.sql import functions as F

### Set variables

In [0]:
metsecret = "8bf4b01a-1d24-4282-b596-4700de5b47f0"
metclientid = "9a30dab6-16a7-4769-bfab-db1162169990"

In [0]:
BASE_URL = "https://frost.met.no"
OBSERVATIONS_URL = f"{BASE_URL}/observations/v0.jsonld"

SECRET_SCOPE = "kv-dbw-dariusz"
SECRET_KEY = "metclientid"

CATALOG = "dbw_dariusz"
SCHEMA = "bronze"
TABLE = "met_norway_frost_observations"

# Example station: Oslo–Blindern. Replace with your required station IDs.
SOURCES = ["SN18700"]

ELEMENTS = [
    "air_temperature",
    "wind_speed",
    "sum(precipitation_amount PT1H)",
]

# Frost defaults to the official time series (ID 0).
# Add filters here if required. Keep quality codes unfiltered for ingestion.
API_FILTERS = {
    "timeseriesids": "0",
}

# UTC interval: START_DATE inclusive, END_DATE exclusive.
# Set both to None for a rolling reload, or use dates such as "2026-01-01".
START_DATE = None
END_DATE = None
LOOKBACK_DAYS = 7

REQUEST_TIMEOUT = (15, 120)
REQUEST_PAUSE_SECONDS = 0.2
MAX_PAGES_PER_CHUNK = 1000

RUN_ID = str(uuid.uuid4())

ROW_SCHEMA = """
    observation_key STRING,
    source_id STRING,
    reference_time TIMESTAMP,
    reference_date DATE,
    element_id STRING,
    value STRING,
    unit STRING,
    quality_code STRING,
    observation_json STRING,
    request_url STRING,
    ingested_at TIMESTAMP,
    ingestion_run_id STRING
"""

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("met_norway_frost")

### Common functions

In [0]:
def canonical_json(value):
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    )


def parse_utc(value):
    parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    if parsed.tzinfo is None:
        raise ValueError(f"Expected a timezone in referenceTime: {value}")
    return parsed.astimezone(timezone.utc)


def format_utc(value):
    return value.strftime("%Y-%m-%dT%H:%M:%SZ")


def resolve_interval():
    if (START_DATE is None) != (END_DATE is None):
        raise ValueError("Set both START_DATE and END_DATE, or leave both as None.")

    if START_DATE is not None:
        start = datetime.strptime(START_DATE, "%Y-%m-%d").replace(
            tzinfo=timezone.utc
        )
        end = datetime.strptime(END_DATE, "%Y-%m-%d").replace(
            tzinfo=timezone.utc
        )
    else:
        if LOOKBACK_DAYS <= 0:
            raise ValueError("LOOKBACK_DAYS must be positive.")
        end = datetime.now(timezone.utc).replace(
            hour=0, minute=0, second=0, microsecond=0
        )
        start = end - timedelta(days=LOOKBACK_DAYS)

    if start >= end:
        raise ValueError("START_DATE must be earlier than END_DATE.")

    return start, end


def create_session(client_id):
    retry = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        respect_retry_after_header=True,
    )

    session = requests.Session()
    session.auth = (client_id, "")
    session.headers.update({
        "Accept": "application/json",
        "User-Agent": "databricks-met-norway-frost-ingestion/1.0",
    })
    session.mount("https://", HTTPAdapter(max_retries=retry))
    return session


def fetch_pages(session, params):
    url = OBSERVATIONS_URL
    query = params
    visited = set()

    for page_number in range(MAX_PAGES_PER_CHUNK):
        # Validate every pagination destination before sending credentials.
        parsed = urlparse(url)
        if (
            parsed.scheme != "https"
            or parsed.netloc != "frost.met.no"
            or parsed.username is not None
        ):
            raise RuntimeError("Unexpected Frost pagination destination.")

        if page_number:
            time.sleep(REQUEST_PAUSE_SECONDS)

        with session.get(
            url,
            params=query,
            timeout=REQUEST_TIMEOUT,
            allow_redirects=False,
        ) as response:
            if response.url in visited:
                raise RuntimeError("Repeated Frost pagination URL.")
            visited.add(response.url)

            # Frost documents 412 as no expected data for the selection.
            if response.status_code == 412 and page_number == 0:
                logger.warning("No expected data for selection: %s", params)
                return

            if response.status_code != 200:
                raise RuntimeError(
                    f"Frost returned HTTP {response.status_code}: "
                    f"{response.text[:1000]}"
                )

            payload = response.json()
            data = payload.get("data")
            if not isinstance(data, list):
                raise RuntimeError("Frost response is missing a valid data array.")

            yield data, response.url

            next_link = payload.get("nextLink")
            if not next_link:
                return

            url = urljoin(response.url, next_link)
            query = None

    raise RuntimeError("Exceeded MAX_PAGES_PER_CHUNK; reduce the request size.")


def fetch_rows(session, source, chunk_start, chunk_end):
    params = {
        **API_FILTERS,
        "sources": source,
        "elements": ",".join(ELEMENTS),
        "referencetime": (
            f"{format_utc(chunk_start)}/{format_utc(chunk_end)}"
        ),
    }

    rows_by_key = {}
    ingested_at = datetime.now(timezone.utc)

    for groups, request_url in fetch_pages(session, params):
        for group in groups:
            source_id = group["sourceId"]
            reference_time = parse_utc(group["referenceTime"])

            # Enforce half-open windows locally, including at API boundaries.
            if not chunk_start <= reference_time < chunk_end:
                continue

            for observation in group["observations"]:
                # Include dimensions that distinguish parallel observations.
                identity = {
                    "source_id": source_id,
                    "reference_time": reference_time.isoformat(),
                    "element_id": observation["elementId"],
                    "time_offset": observation.get("timeOffset"),
                    "time_resolution": observation.get("timeResolution"),
                    "time_series_id": str(
                        observation.get("timeSeriesId", "0")
                    ),
                    "level": observation.get("level"),
                }

                key = hashlib.sha256(
                    canonical_json(identity).encode("utf-8")
                ).hexdigest()

                raw_json = canonical_json(observation)
                previous = rows_by_key.get(key)

                if previous is not None and previous[8] != raw_json:
                    raise RuntimeError(
                        f"Conflicting observations for key {key}; "
                        "review the source response."
                    )

                value = observation.get("value")
                quality = observation.get("qualityCode")

                rows_by_key[key] = (
                    key,
                    source_id,
                    reference_time,
                    reference_time.date(),
                    observation["elementId"],
                    None if value is None else str(value),
                    observation.get("unit"),
                    None if quality is None else str(quality),
                    raw_json,
                    request_url,
                    ingested_at,
                    RUN_ID,
                )

    return list(rows_by_key.values())


def merge_rows(rows, target_table):
    if not rows:
        return 0

    batch = spark.createDataFrame(rows, schema=ROW_SCHEMA)

    (
        DeltaTable.forName(spark, target_table)
        .alias("target")
        .merge(
            batch.alias("source"),
            "target.observation_key = source.observation_key",
        )
        .whenMatchedUpdateAll(
            condition=(
                "NOT (target.observation_json "
                "<=> source.observation_json)"
            )
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    return len(rows)

### Main code

In [0]:
# Validate identifiers before interpolating them into SQL.
for identifier in [CATALOG, SCHEMA, TABLE]:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier):
        raise ValueError(f"Invalid destination identifier: {identifier}")

if not SOURCES or not ELEMENTS:
    raise ValueError("Configure at least one source and one element.")

if any(not re.fullmatch(r"SN\d+(?::\d+)?", source) for source in SOURCES):
    raise ValueError("Use explicit Frost station IDs, for example SN18700.")

target_table = f"{CATALOG}.{SCHEMA}.{TABLE}"
quoted_schema = f"`{CATALOG}`.`{SCHEMA}`"
quoted_table = f"{quoted_schema}.`{TABLE}`"

spark.conf.set("spark.sql.session.timeZone", "UTC")

start_time, end_time = resolve_interval()

# The catalog must already exist and the job identity needs write permissions.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {quoted_schema}")
spark.sql(
    f"CREATE TABLE IF NOT EXISTS {quoted_table} "
    f"({ROW_SCHEMA}) USING DELTA"
)
# Temporary solution as KeyVault needs to be set up
client_id = metclientid

if not client_id.strip():
    raise ValueError("The Frost client ID secret is empty.")

total_processed = 0

with create_session(client_id) as session:
    chunk_start = start_time

    while chunk_start < end_time:
        chunk_end = min(chunk_start + timedelta(days=1), end_time)

        for source in dict.fromkeys(SOURCES):
            rows = fetch_rows(session, source, chunk_start, chunk_end)
            processed = merge_rows(rows, target_table)
            total_processed += processed

            logger.info(
                "source=%s start=%s end=%s observations=%s",
                source,
                format_utc(chunk_start),
                format_utc(chunk_end),
                processed,
            )

            time.sleep(REQUEST_PAUSE_SECONDS)

        chunk_start = chunk_end

logger.info(
    "Completed run_id=%s observations_processed=%s target=%s",
    RUN_ID,
    total_processed,
    target_table,
)
'''
display(
    spark.table(target_table)
    .where(
        (F.col("reference_time") >= F.lit(start_time))
        & (F.col("reference_time") < F.lit(end_time))
        & F.col("element_id").isin(ELEMENTS)
    )
    .orderBy(F.col("reference_time").desc())
    .limit(100)
)
'''